In [1]:
import osmnx as ox
import networkx as nx
import geopandas as gpd
import pandas as pd
from shapely.geometry import box
import pyproj
from shapely.ops import transform

#To form the multi-modal network, we use the OSM walking network for Luxembourg, our GTFS data for the PT network and then a connector layer .


#Load stops into epfs 169
stops = gpd.read_file(r"C:\Users\luca\Downloads\Lux_public_transport_project\stop_freq_avl.gpkg")
stops = stops.to_crs(epsg=2169)

#Walking speed around 5 km/h
walking_speed = 1.4  # m/s

#Build walking network over the bounding box of AVL stops only. Similar to P+R stop OSM query
minx, miny, maxx, maxy = stops.total_bounds
buffer_m = 300

bbox_geom = box(minx - buffer_m, miny - buffer_m,
                maxx + buffer_m, maxy + buffer_m)
proj = pyproj.Transformer.from_crs(2169, 4326, always_xy=True).transform
bbox_wgs84 = transform(proj, bbox_geom)
walking_network = ox.graph_from_polygon(bbox_wgs84, network_type="walk")

#Take largest connected component and project it to EPSG:2169
walking_network = walking_network.subgraph(
    max(nx.connected_components(walking_network.to_undirected()), key=len)
).copy()
walking_network = ox.project_graph(walking_network, to_crs="EPSG:2169")
stops["x"] = stops.geometry.x
stops["y"] = stops.geometry.y

#Relabel nodes to distinguish between the PT network and walking network
walk_mapping = {node: f"walk_{node}" for node in walking_network.nodes}
walking_network = nx.relabel_nodes(walking_network, walk_mapping)

#Assign travel time weights in minutes
for u, v, k, data in walking_network.edges(keys=True, data=True):
    data["edge_type"] = "walk"
    data["mode"] = "walk"
    data["weight"] = (data["length"] / walking_speed) / 60  # minutes

print("Walking network created")


Walking network created


In [2]:
#Next, we create the PT graph

#Load trips and stop times and headways
trips      = pd.read_csv("trips.csv", dtype={"route_id": str})
stop_times = pd.read_csv("stop_times.csv", dtype={"stop_id": str})
headways  = pd.read_csv("stop_route_headways.csv", dtype={"stop_id": str, "route_id": str})

#Assign a route ID for each trip ID. This distinguishes bus lines
trip_route = trips.set_index("trip_id")["route_id"].to_dict()

#Create route=headway dict. Each stop has a wait time of half the average headway of the bus line.
#Fallback time of five minutes
stop_wait = headways.assign(wait_time=lambda df: df["avg_headway"] / 2).set_index(["stop_id", "route_id"])["wait_time"].to_dict()
avg_wait_time = 5  

#Create graph
pt_network = nx.MultiDiGraph()

#Maps trip Id to route ID
stop_route_pairs = stop_times[["trip_id", "stop_id"]].assign(
    route_id=lambda x: x["trip_id"].map(trip_route))[["stop_id", "route_id"]].drop_duplicates()

#Attach to the general stop dataframe
node_df = stop_route_pairs.merge(stops[["stop_id", "x", "y", "stop_name"]], on="stop_id", how="left")


#Two levels of nodes:
#  stop nodes       — attach origins/destinations for routing
#  stop_route nodes — line-level nodes for boarding/alighting

#Edges go like this stop1 -> stop1_route1 -> stop2_route1 -> stop2 -> stop2_route2 etc.

#Stop layer 
stop_nodes = [
    (f"pt_{row.stop_id}", {"stop_id": row.stop_id, "x": row.x, "y": row.y,
                            "name": row.stop_name, "node_type": "stop"})
    for row in node_df.drop_duplicates(subset="stop_id").itertuples()
]
pt_network.add_nodes_from(stop_nodes)

#Stop_route layer
stop_route_nodes = [
    (f"pt_{row.stop_id}_{row.route_id}", {"stop_id": row.stop_id, "route_id": row.route_id,
                                           "x": row.x, "y": row.y, "name": row.stop_name,
                                           "node_type": "stop_route"})
    for row in node_df.itertuples()
]
pt_network.add_nodes_from(stop_route_nodes)

# Stop <-> stop_route connection
#From stop to stop_route gives wait time per route but from stop_route to stop no weight 
boarding_edges = [
    (f"pt_{row.stop_id}", f"pt_{row.stop_id}_{row.route_id}",
     {"weight": stop_wait.get((row.stop_id, row.route_id), avg_wait_time),
      "edge_type": "board", "mode": "board"})
    for row in node_df.itertuples()
]

alighting_edges = [
    (f"pt_{row.stop_id}_{row.route_id}", f"pt_{row.stop_id}",
     {"weight": 0, "edge_type": "alight", "mode": "alight"}) 
    for row in node_df.itertuples()
]
pt_network.add_edges_from(boarding_edges)
pt_network.add_edges_from(alighting_edges)

#stop_route <-> stop_route connections, weight based on travel time
st      = stop_times.sort_values(["trip_id", "stop_sequence"])
st_next = st.groupby("trip_id").shift(-1)
edges_df = st.join(st_next, rsuffix="_next").dropna(subset=["stop_id_next"])

#Edges df has trip, stop and route id as well as the next stop and departure and arrival times
edges_df = edges_df[["trip_id", "stop_id", "stop_id_next", "dep_sec", "arr_sec_next"]]
edges_df["route_id"] = edges_df["trip_id"].map(trip_route)

#Travel time is based on stop_times df. Using stop sequence we can subtract the -1 stops from the 0 stop to find the time it takes to go from one stop to the other.
edges_df["travel_time_sec"] = edges_df["arr_sec_next"] - edges_df["dep_sec"]

edges_df = edges_df[edges_df["travel_time_sec"] > 0]
edges_df["weight"] = edges_df["travel_time_sec"] / 60  # minutes

#Finally we add the edges df to the lift
attr_dicts = edges_df[["weight", "trip_id", "route_id"]].to_dict("records")
for d in attr_dicts:
    d["edge_type"] = "pt"
    d["mode"]      = "bus"

edges = list(zip(
    "pt_" + edges_df.stop_id.astype(str)      + "_" + edges_df.route_id.astype(str),
    "pt_" + edges_df.stop_id_next.astype(str) + "_" + edges_df.route_id.astype(str),
    edges_df.trip_id,
    attr_dicts
))
pt_network.add_edges_from(edges)
print("PT network created")


PT network created


In [3]:
#Create connector edges between walking and PT networks i.e. walk_12323 <-> stop_23i32i
connector_graph = nx.MultiDiGraph()

#Find the nearest walking node to each AVL stop
stops["node"] = ox.distance.nearest_nodes(walking_network, stops.geometry.x, stops.geometry.y)

#Create one big df with walking nodes and pt_nodes
connector_df = node_df[["stop_id", "route_id"]].drop_duplicates().merge(
    stops[["stop_id", "node"]], on="stop_id", how="left")

connector_df["pt_node"]   = "pt_" + connector_df["stop_id"]
connector_df["walk_node"] = connector_df["node"]

#Walk -> PT edges 
walk_to_pt_edges = [
    (row.walk_node, row.pt_node, {"weight": 0, "edge_type": "connector", "mode": "pt_walk_connect"})
    for row in connector_df.itertuples()
]
connector_graph.add_edges_from(walk_to_pt_edges)

#PT -> walk edges (
pt_to_walk_edges = [
    (row.pt_node, row.walk_node, {"weight": 0, "edge_type": "connector", "mode": "walk_pt_connect"})
    for row in connector_df.itertuples()
]
connector_graph.add_edges_from(pt_to_walk_edges)
print("Connector graph created")

Connector graph created


In [4]:
#Compose all three graphs into the final multi-modal network
import pickle

multimodal_network = nx.compose_all([walking_network, pt_network, connector_graph])

#We analyse the network in 2169
multimodal_network.graph["crs"] = "EPSG:2169"

with open("multimodal_network.pkl", "wb") as f:
    pickle.dump(multimodal_network, f)

print("Multi-modal graph created")

Multi-modal graph created


In [5]:
#Validation checks
print(f"Nodes: {multimodal_network.number_of_nodes()}")
print(f"Edges: {multimodal_network.number_of_edges()}")

#Check graph is fully connected
components = list(nx.weakly_connected_components(multimodal_network))
print(f"Weakly connected components: {len(components)}")
print(f"Largest: {len(max(components, key=len))} nodes")

#Check for isolated nodes
isolates = list(nx.isolates(multimodal_network))
print(f"Isolated nodes: {len(isolates)}")

#PT stop check
test_node = "pt_200405020_3646"
print(f"Test node in graph: {test_node in multimodal_network}")
print(f"Degree: {multimodal_network.degree(test_node)}")


Nodes: 29011
Edges: 92205
Weakly connected components: 1
Largest: 29011 nodes
Isolated nodes: 0
Test node in graph: True
Degree: 17
